# Tugas Akhir Praktikum Logical Agents: Wumpus World 5×5

## Identitas
- **Nama**   : Wayan Raditya Putra  
- **NRP**    : 5054241029  
- **Departemen** : Teknik Informatika  
- **Prodi** : RKA
- **Mata Kuliah** : Kecerdasan Komputasional
- **Dosen Pengampu** :  
  - Prof. Dr. Eng. Nanik Suciati, S.Kom., M.Kom.  
  - Imam Mustafa Kamal, S.ST, Ph.D.  

---

## Deskripsi Tugas
Agen ditempatkan pada lingkungan **Wumpus World 5×5**.  
- Posisi awal agen: `[1,1]`  
- Posisi emas: `[2,4]`  
- Posisi pit: `(1,3), (1,5), (4,1), (4,5)`  
- Posisi Wumpus: `(3,1)`  

Aturan Wumpus World:  
- Pit menimbulkan **breeze** di sel tetangga (atas, bawah, kiri, kanan).  
- Wumpus menimbulkan **stench** di sel tetangga (atas, bawah, kiri, kanan).  

---

## Tujuan
1. Mendefinisikan proposisi dan aturan logika (R1–Rn) berdasarkan aturan Wumpus World.  
2. Menyusun rangkaian proposisi secara sistematis dari posisi awal hingga emas.  
3. Melakukan inferensi menggunakan **Truth Table Entailment (TT-entails)** untuk memverifikasi kebenaran inferensi.  
4. Melakukan inferensi menggunakan **Forward Chaining** (NRP ganjil).  
5. Membuktikan apakah agen dapat mencapai emas dengan aman, serta menuliskan jalur langkah demi langkah.  
6. Mengimplementasikan program Python yang merepresentasikan inferensi logika agen.  
---

## Pendahuluan

Pada bidang **Kecerdasan Buatan (Artificial Intelligence)**, salah satu pendekatan yang digunakan untuk 
merepresentasikan pengetahuan adalah melalui **Logical Agents**. Logical agent merupakan agen yang 
mengambil keputusan berdasarkan **aturan logika** dan inferensi, bukan sekadar trial-and-error.  
Agen jenis ini cocok untuk lingkungan yang penuh ketidakpastian tetapi memiliki aturan formal yang 
jelas, seperti **Wumpus World**.

**Wumpus World** adalah dunia berbentuk grid yang dipopulerkan dalam literatur AI (Russell & Norvig). 
Lingkungan ini digunakan untuk menguji kemampuan agen dalam bernavigasi secara aman untuk mencapai 
tujuan (misalnya menemukan emas), sambil menghindari bahaya berupa **pit** (lubang) dan **wumpus** 
(monster). Agen hanya bisa merasakan indikator lingkungan:  
- **Breeze** → muncul di sel yang berdekatan dengan pit.  
- **Stench** → muncul di sel yang berdekatan dengan wumpus.  

Untuk dapat bernavigasi dengan aman, agen harus mampu:  
1. Mendefinisikan proposisi yang merepresentasikan kondisi dunia.  
2. Menggunakan aturan logika (R1–Rn) untuk melakukan inferensi.  
3. Menentukan langkah berikutnya berdasarkan inferensi yang sahih.

Dalam tugas ini digunakan dua metode inferensi utama:  
- **Truth Table Entailment (TT-entails)**: metode dasar dengan membangun tabel kebenaran untuk 
  memverifikasi apakah suatu proposisi logis benar secara konsisten.  
- **Forward Chaining**: metode berbasis aturan produksi yang mulai dari fakta yang diketahui 
  kemudian menurunkan fakta-fakta baru sampai mencapai kesimpulan.  

Dengan pendekatan ini, agen diharapkan dapat membuktikan apakah emas di koordinat `(2,4)` dapat 
dicapai dari posisi awal `(1,1)` dengan aman sesuai aturan Wumpus World.


## Definisi Proposisi & Aturan Umum (R1–Rn)

### Proposisi
1. **P(x,y)** : Ada *pit* pada sel `(x,y)`  
2. **W(x,y)** : Ada *wumpus* pada sel `(x,y)`  
3. **G(x,y)** : Ada emas (*gold*) pada sel `(x,y)`  
4. **B(x,y)** : Ada *breeze* pada sel `(x,y)`  
5. **S(x,y)** : Ada *stench* pada sel `(x,y)`  
6. **OK(x,y)** : Sel `(x,y)` aman untuk dimasuki agen  
7. **¬OK(x,y)** : Sel `(x,y)` berbahaya (ada kemungkinan pit atau wumpus) 

---

### Aturan Umum (R1–Rn)
**R1 (Breeze Rule)**  
B(x,y) ↔ (P(x-1,y) ∨ P(x+1,y) ∨ P(x,y-1) ∨ P(x,y+1))

**R2 (Stench Rule)**  
S(x,y) ↔ (W(x-1,y) ∨ W(x+1,y) ∨ W(x,y-1) ∨ W(x,y+1))
  
**R3 (Safety Rule)**  
  ¬B(x,y) ∧ ¬S(x,y) → OK(x,y)
  
**R4 (Pit Inference)**  
Jika ada breeze di `(x,y)` dan semua tetangga selain `(i,j)` sudah terbukti aman,  
maka tetangga `(i,j)` pasti pit.  

**Notasi:**
B(x,y) ∧ (OK(t₁) ∧ OK(t₂) ∧ ... ∧ OK(tₙ)) → P(i,j)  

dengan:  
- `(t₁, t₂, ..., tₙ)` = semua tetangga dari `(x,y)` **kecuali** `(i,j)`  
- `(i,j)` = satu-satunya tetangga yang belum aman  

**R5 (Wumpus Inference)**  
Jika ada stench di `(x,y)` dan semua tetangga selain `(i,j)` sudah terbukti aman,  
maka tetangga `(i,j)` pasti wumpus.  

**Notasi:**
S(x,y) ∧ (OK(t₁) ∧ OK(t₂) ∧ ... ∧ OK(tₙ)) → W(i,j)  

dengan:  
- `(t₁, t₂, ..., tₙ)` = semua tetangga dari `(x,y)` **kecuali** `(i,j)`  
- `(i,j)` = satu-satunya tetangga yang belum aman  


**R6 (Gold Detection)**  
Jika ada emas di `(x,y)`, maka `(x,y)` adalah tujuan agen.  

**Notasi:**
G(x,y) → Goal(x,y)

## Project Setup

Pada tahap ini dilakukan proses **import library** dan modul pendukung yang dibutuhkan untuk mengerjakan tugas Logical Agents pada Wumpus World.  
Library yang digunakan terdiri dari modul internal (`utils.py`, `logic.py`, `agents.py`) maupun library eksternal Python.

### Penjelasan Library

* **utils** → berisi fungsi utilitas tambahan yang mendukung proses inferensi/logika.
* **logic** → berisi implementasi representasi logika (proposisi, aturan, inferensi).
* **agents** → modul berisi definisi agen dan cara agen berinteraksi dengan Wumpus World.
* **math** → library standar Python untuk operasi matematika.
* **inspect.getsource** → digunakan untuk menampilkan kode sumber dari fungsi tertentu (berguna saat analisis).
* **IPython.display.HTML** → menampilkan output HTML di Jupyter Notebook.
* **tabulate** → menghasilkan tabel rapi untuk menyajikan hasil inferensi dan analisis.



In [66]:
from utils import *
from logic import *
import agents
import math
from inspect import getsource
from IPython.display import HTML
from tabulate import tabulate

## Representasi Peta dan Informasi Wumpus World

Kelas `WumpusAgent` digunakan sebagai **alat bantu** untuk membangun representasi dunia Wumpus 
dalam bentuk grid (map) serta memasukkan informasi lingkungan ke dalam *knowledge base* (KB) agen.

- **World (Peta 5×5)** direpresentasikan sebagai list of list, di mana setiap cell dapat berisi:
  - `"P"` → Pit
  - `"W"` → Wumpus
  - `"G"` → Gold
  - `"B"` → Breeze
  - `"S"` → Stench
  - `[]`   → Kosong (tidak ada percept)

- **Knowledge Base (KB)** dibangun secara bertahap:
  1. Agen memulai dari posisi awal `(1,1)` → otomatis dianggap aman (`Safe(1,1)`).
  2. Agen membaca **percepts** di cell tersebut (misalnya Breeze, Stench).
  3. Informasi percepts dimasukkan ke dalam KB sebagai proposisi logika, contohnya:
     - `Breeze(1,2)`
     - `Stench(2,1)`
     - `Gold(2,4)`

Dengan cara ini, agen dapat menghubungkan **peta (map)** dan **pengetahuan logika** yang 
dibutuhkan untuk melakukan inferensi, sehingga jalur menuju emas dapat ditentukan secara aman.


In [67]:
class AgenWumpus:
    def __init__(self, dunia, posisi_awal, berhenti_jika_ketemu=True):
        posisi_awal = tuple(posisi_awal)
        self.dunia = dunia
        self.pengetahuan = PropKB()
        self.sudah_dikunjungi = set()
        self.berhenti_jika_ketemu = berhenti_jika_ketemu
        self.posisi_sekarang = posisi_awal
        self.arah = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # Atas, Kanan, Bawah, Kiri
        self.ada_emas = False
        self.lokasi_emas = None

        # Tandai titik awal sebagai aman
        self.pengetahuan.tell(expr(f"Aman({posisi_awal[0]},{posisi_awal[1]})"))
        self.sudah_dikunjungi.add(posisi_awal)
        self.perbarui_pengetahuan(posisi_awal[0], posisi_awal[1])

    def ambil_persepsi(self, x, y):
        koordinat_x = len(self.dunia) - y
        koordinat_y = x - 1
        if 0 <= koordinat_x < len(self.dunia) and 0 <= koordinat_y < len(self.dunia[0]):
            return set(self.dunia[koordinat_x][koordinat_y])
        return set()

    def perbarui_pengetahuan(self, x, y):
        persepsi = self.ambil_persepsi(x, y)
        self.pengetahuan.tell(expr(f"Aman({x},{y})"))
        print(f"Perbarui pengetahuan untuk ({x},{y}):")
        for tanda in persepsi:
            self.pengetahuan.tell(expr(f"{tanda}({x},{y})"))
            print(f"  - Tambah {tanda}({x},{y}) ke basis pengetahuan")
            if tanda == "B":
                ekspresi_tetangga = " | ".join([f"Lubang({nx},{ny})" for nx, ny in self.tetangga(x, y)])
                self.pengetahuan.tell(expr(ekspresi_tetangga))
                print(f"  - Tambah {ekspresi_tetangga} karena ada Angin")
            if tanda == "S":
                ekspresi_tetangga = " | ".join([f"Wumpus({nx},{ny})" for nx, ny in self.tetangga(x, y)])
                self.pengetahuan.tell(expr(ekspresi_tetangga))
                print(f"  - Tambah {ekspresi_tetangga} karena ada Bau")
            if tanda == "G":
                self.ada_emas = True
                self.lokasi_emas = (x, y)
                print(f"Emas ditemukan di ({x},{y})!")

        # Update pengetahuan tetangga
        for nx, ny in self.tetangga(x, y):
            tanda_tetangga = self.ambil_persepsi(nx, ny)
            if not tanda_tetangga:
                self.pengetahuan.tell(expr(f"Aman({nx},{ny})"))
                print(f"  - Tambah Aman({nx},{ny}) ke basis pengetahuan (tetangga)")
            for tanda in tanda_tetangga:
                self.pengetahuan.tell(expr(f"{tanda}({nx},{ny})"))
                print(f"  - Tambah {tanda}({nx},{ny}) ke basis pengetahuan (tetangga)")

    def aman(self, x, y):
        if expr(f"Lubang({x},{y})") in self.pengetahuan.clauses:
            return False, "ada lubang"
        if expr(f"Wumpus({x},{y})") in self.pengetahuan.clauses:
            return False, "ada wumpus"
        if (expr(f"Lubang({x},{y})") not in self.pengetahuan.clauses and 
            expr(f"Wumpus({x},{y})") not in self.pengetahuan.clauses) or (
            self.pengetahuan.ask_if_true(expr(f"Aman({x},{y})"))):
            return True, "diketahui aman"
        if self.pengetahuan.ask_if_true(expr(f"Lubang({x},{y})")):
            return False, "mungkin ada lubang"
        if self.pengetahuan.ask_if_true(expr(f"Wumpus({x},{y})")):
            return False, "mungkin ada wumpus"
        return True, "tidak ada bahaya yang pasti di basis pengetahuan"

    def tetangga(self, x, y):
        return [(x + dx, y + dy) for dx, dy in self.arah
                if 1 <= x + dx <= len(self.dunia[0]) and 1 <= y + dy <= len(self.dunia)]

    def jelajah(self):
        print("Pencarian dengan DFS:")
        print(f"Mulai dari posisi ({self.posisi_sekarang[0]},{self.posisi_sekarang[1]})")
        tumpukan = []
        langkah = 1
        for nx, ny in self.tetangga(self.posisi_sekarang[0], self.posisi_sekarang[1]):
            aman, alasan = self.aman(nx, ny)
            if aman:
                tumpukan.append((nx, ny))
                print(f"  Tambah ({nx},{ny}) ke tumpukan karena {alasan}")

        while tumpukan:
            print(f"\nLangkah {langkah}")
            langkah += 1
            print(f"Tumpukan: {tumpukan}")
            x, y = tumpukan.pop()
            print(f"\nPertimbangkan posisi ({x},{y}):")
            aman, alasan = self.aman(x, y)
            if (x, y) not in self.sudah_dikunjungi and aman:
                print(f"  Mengunjungi ({x},{y}) karena {alasan}")
                self.sudah_dikunjungi.add((x, y))
                self.posisi_sekarang = (x, y)
                self.perbarui_pengetahuan(x, y)

                print(f"Selesai kunjungi ({x},{y})")
                print(f"Basis pengetahuan sekarang: {self.pengetahuan.clauses}")

                if self.berhenti_jika_ketemu and self.ada_emas:
                    break

                tetangga_pos = self.tetangga(x, y)
                tetangga_pos.sort(key=lambda pos: (pos[1], pos[0]), reverse=True)

                print("Pertimbangkan sel tetangga:")
                for next_x, next_y in tetangga_pos:
                    print(f"Evaluasi ({next_x},{next_y}):")
                    if (next_x, next_y) in self.sudah_dikunjungi:
                        print(f"  Lewati ({next_x},{next_y}) karena sudah dikunjungi")
                    else:
                        aman, alasan = self.aman(next_x, next_y)
                        if aman:
                            tumpukan.append((next_x, next_y))
                            print(f"  Tambah ({next_x},{next_y}) ke tumpukan karena {alasan}")
                        else:
                            print(f"  Lewati ({next_x},{next_y}) karena {alasan}")
            else:
                if (x, y) in self.sudah_dikunjungi:
                    print(f"  Lewati ({x},{y}) karena sudah dikunjungi")
                else:
                    print(f"  Lewati ({x},{y}) karena {alasan}")

        if self.ada_emas:
            print(f"\nPencarian selesai. Emas ditemukan di ({self.lokasi_emas[0]},{self.lokasi_emas[1]}).")
        else:
            print("\nPencarian selesai. Emas tidak ditemukan.")
        print(f"Posisi akhir: ({self.posisi_sekarang[0]},{self.posisi_sekarang[1]})")


In [68]:
world = [
    # y = 5
    [["P"], ["B"], ["B"], ["P"], ["B"]],   # row 5
    # y = 4
    [["B"], ["G"], [], ["B"], []],      # row 4
    # y = 3
    [["P"], ["B"], [], [], []],            # row 3
    # y = 2
    [["B"], [], ["S"], ["B"], []],         # row 2
    # y = 1
    [[], ["S"], ["W"], ["P"], ["B"]]     # row 1
]






In [69]:
# Agen pakai TT entails
agen_tt = AgenWumpus(world, posisi_awal=(1,1))
agen_tt.jelajah()


Perbarui pengetahuan untuk (1,1):
  - Tambah B(1,2) ke basis pengetahuan (tetangga)
  - Tambah S(2,1) ke basis pengetahuan (tetangga)
Pencarian dengan DFS:
Mulai dari posisi (1,1)
  Tambah (1,2) ke tumpukan karena diketahui aman
  Tambah (2,1) ke tumpukan karena diketahui aman

Langkah 1
Tumpukan: [(1, 2), (2, 1)]

Pertimbangkan posisi (2,1):
  Mengunjungi (2,1) karena diketahui aman
Perbarui pengetahuan untuk (2,1):
  - Tambah S(2,1) ke basis pengetahuan
  - Tambah Wumpus(2,2) | Wumpus(3,1) | Wumpus(1,1) karena ada Bau
  - Tambah Aman(2,2) ke basis pengetahuan (tetangga)
  - Tambah W(3,1) ke basis pengetahuan (tetangga)
  - Tambah Aman(1,1) ke basis pengetahuan (tetangga)
Selesai kunjungi (2,1)
Basis pengetahuan sekarang: [Aman(1, 1), Aman(1, 1), B(1, 2), S(2, 1), Aman(2, 1), S(2, 1), (Wumpus(2, 2) | Wumpus(3, 1) | Wumpus(1, 1)), Aman(2, 2), W(3, 1), Aman(1, 1)]
Pertimbangkan sel tetangga:
Evaluasi (2,2):
  Tambah (2,2) ke tumpukan karena diketahui aman
Evaluasi (3,1):
  Tambah (3,1) 

In [70]:

# Sumber = Aima Data, Class wumpus , agent, 
class WumpusAgent:
    def __init__(self, world, initial_pos=(1,1)):
        self.world = world
        self.kb = PropKB()
        self.visited = set()
        self.position = initial_pos

        # Cell Agent pertama kali
        self.kb.tell(expr(f"Safe({initial_pos[0]},{initial_pos[1]})"))
        self.visited.add(initial_pos)

        # Update KB berdasarkan percepts
        self.update_kb(*initial_pos)

    def get_percepts(self, x, y):
        """
        Ambil percept dari koordinat (x,y) sesuai world
        """
        world_x = len(self.world) - y
        world_y = x - 1
        if 0 <= world_x < len(self.world) and 0 <= world_y < len(self.world[0]):
            return set(self.world[world_x][world_y])
        return set()

    # … di dalam class WumpusAgent yang sudah ada …

        # … di dalam class WumpusAgent yang sudah ada …

    def update_kb(self, x, y):
        """
        Update KB dari percepts posisi (x,y):
        - Simpan Breeze/Stench/Gold jika ada
        - Simpan NoBreeze/NoStench jika tidak ada (penting untuk inferensi)
        - Tandai sel saat ini aman (agen berada di situ & tidak mati)
        """
        percepts = self.get_percepts(x, y)
        hasB = ("B" in percepts) or ("Breeze" in percepts)
        hasS = ("S" in percepts) or ("Stench" in percepts)
        hasG = ("G" in percepts) or ("Gold" in percepts)

        if hasB: self.kb.tell(expr(f"Breeze({x},{y})"))
        else:    self.kb.tell(expr(f"NoBreeze({x},{y})"))

        if hasS: self.kb.tell(expr(f"Stench({x},{y})"))
        else:    self.kb.tell(expr(f"NoStench({x},{y})"))

        if hasG: self.kb.tell(expr(f"Gold({x},{y})"))

        # Agen hidup di sel ini ⇒ sel ini aman
        self.kb.tell(expr(f"Safe({x},{y})"))
        self.visited.add((x,y))


In [71]:
world = [
    # y = 5
    [["P"], ["B"], ["B"], ["P"], ["B"]],   # row 5
    # y = 4
    [["B"], ["G"], [], ["B"], []],      # row 4
    # y = 3
    [["P"], ["B"], [], [], []],            # row 3
    # y = 2
    [["B"], [], ["S"], ["B"], []],         # row 2
    # y = 1
    [[], ["S"], ["W"], ["P"], ["B"]]     # row 1
]

agent = WumpusAgent(world, initial_pos=(1,1))

In [72]:
import time
import random
from functools import reduce
from logic import expr, tt_entails, pl_fc_entails, PropKB

# Helper: conjoin semua klausa jadi 1 Expr
def conjoin(clauses):
    return reduce(lambda a, b: a & b, clauses)

# Helper: filter KB hanya untuk cell tertentu
def clauses_for_cells(clauses, cells):
    cells_str = {f"({x},{y})" for (x,y) in cells}
    kept = []
    for c in clauses:
        symbols = [str(s) for s in c.args] if hasattr(c, "args") else [str(c)]
        if any(any(cs in tok for cs in cells_str) for tok in symbols):
            kept.append(c)
    return kept

def tt_entails_local(agent, query, scope_cells):
    local = clauses_for_cells(agent.kb.clauses, scope_cells)
    if not local:  # fallback kalau kosong
        return False
    kb_expr = conjoin(local)
    return tt_entails(kb_expr, query)

def pl_fc_entails_simple(kb, query):
    # naive forward chaining
    agenda = list(kb.clauses)
    inferred = set()
    while agenda:
        fact = agenda.pop()
        if fact == query:
            return True
        if fact not in inferred:
            inferred.add(fact)
    return False


# Cek apakah sel aman dengan dua mode
def entails_safe(agent, nx, ny, mode="fc"):
    q = expr(f"Safe({nx},{ny})")
    if mode == "fc":
        return pl_fc_entails_simple(agent.kb, q)
    elif mode == "tt":
        # gunakan hanya visited + tetangga + sel target
        scope = set(agent.visited) | {(nx,ny)}
        return tt_entails_local(agent, q, scope)
    else:
        raise ValueError("Mode harus 'fc' atau 'tt'")


# Eksplorasi dunia
def explore(agent, mode="fc"):
    path = [agent.position]
    t0 = time.time()

    while True:
        x, y = agent.position
        agent.update_kb(x, y)

        # cek emas
        if "G" in agent.get_percepts(x, y):
            print(f"[{mode.upper()}] GOLD ditemukan di {(x,y)}")
            break

        # kandidat tetangga
        cand = [(nx, ny) for (nx, ny) in [(x+1,y),(x-1,y),(x,y+1),(x,y-1)]
                if 1 <= nx <= len(agent.world[0]) and 1 <= ny <= len(agent.world)]

        cand = [c for c in cand if c not in agent.visited]
        moved = False

        # 1) coba cari safe neighbor
        safe_neighbors = [c for c in cand if entails_safe(agent, c[0], c[1], mode=mode)]
        if safe_neighbors:
            nx, ny = random.choice(safe_neighbors)  # random biar nggak bias kanan
            agent.position = (nx, ny)
            agent.visited.add((nx, ny))
            agent.kb.tell(expr(f"Safe({nx},{ny})"))
            path.append((nx, ny))
            print(f"[{mode.upper()}] → move ke {(nx,ny)} (terbukti aman)")
            moved = True
        else:
            # 2) cautious mode
            possible = [c for c in cand if expr(f"P({c[0]},{c[1]})") not in agent.kb.clauses
                                      and expr(f"W({c[0]},{c[1]})") not in agent.kb.clauses]
            if possible:
                nx, ny = random.choice(possible)
                agent.position = (nx, ny)
                agent.visited.add((nx, ny))
                path.append((nx, ny))
                print(f"[{mode.upper()}] ⚠️ move ke {(nx,ny)} (belum terbukti bahaya, mode CAUTIOUS)")
                moved = True
            else:
                print(f"[{mode.upper()}] Agen benar-benar buntu, stop.")
                break

        if not moved:
            break

    dt = time.time() - t0
    print(f"Waktu eksekusi ({mode.upper()}): {dt:.6f} detik")
    return path


In [73]:
print("=== Forward Chaining ===")
path_fc = explore(agent, mode="fc")

=== Forward Chaining ===
[FC] ⚠️ move ke (2, 1) (belum terbukti bahaya, mode CAUTIOUS)
[FC] ⚠️ move ke (2, 2) (belum terbukti bahaya, mode CAUTIOUS)
[FC] ⚠️ move ke (2, 3) (belum terbukti bahaya, mode CAUTIOUS)
[FC] ⚠️ move ke (2, 4) (belum terbukti bahaya, mode CAUTIOUS)
[FC] GOLD ditemukan di (2, 4)
Waktu eksekusi (FC): 0.001000 detik
